In [2]:
import tensorflow as tf
import os
import glob
import numpy as np
from model import build_1d_cnn_model
from common import INPUT_SHAPE,OUTPUT_DIM_NOTES, fast_gpu_map,parse_example
cnn_model=build_1d_cnn_model(None,INPUT_SHAPE,44,False)
cnn_model.summary()
tf.keras.utils.plot_model(cnn_model,to_file='cnn_model.png',show_shapes=True)
cnn_model.load_weights('/home/gerald/workspace/src/GuitarMidi-LV2/python/neuralnetmodelling/checkpoints/guitarmidi_epoch50_valAcc0.9661_valPrec0.6793_valRecall0.7547.weights.h5')#
#cnn_model.load_weights('guitarmidi.weights.h5')

input_data_dir = '/home/gerald/workspace/src/GuitarMidi-LV2/python/neuralnetmodelling/data_slices'
input_filepaths = '/home/gerald/workspace/src/GuitarMidi-LV2/python/neuralnetmodelling/data_slices/training_subset/filtered_poly_data.tfrecord'#sorted(glob.glob(os.path.join(input_data_dir, '**', 'input', 'data.tfrecord'), recursive=True))
# train_dataset = tf.data.Dataset.from_tensor_slices((input_filepaths))
# train_dataset=train_dataset.shuffle(buffer_size=len(input_filepaths))
# train_dataset=train_dataset.take(100)
# train_dataset = train_dataset.map(tf_load_sample_from_files, num_parallel_calls=tf.data.AUTOTUNE)
def representative_data_gen():
    # Use TFRecordDataset to actually read the files
    # We only need a few samples to calibrate quantization
    raw_dataset = tf.data.TFRecordDataset(input_filepaths).map(parse_example).take(100)
    
    # Map using your existing loading function
    calib_dataset = raw_dataset.map(lambda path: fast_gpu_map(path, training=False))
    
    for input_value, _ in calib_dataset.batch(1):
        # input_value is the (312, 256, 1) tensor
        yield [input_value]

converter = tf.lite.TFLiteConverter.from_keras_model(cnn_model)

converter.optimizations = [ tf.lite.Optimize.DEFAULT]
# converter.representative_dataset = representative_data_gen
# converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
# converter.target_spec.supported_types = [tf.int8] 

# converter.target_spec.supported_types = [tf.float16]
# converter.target_spec._experimental_supported_accumulation_type = tf.dtypes.float16
# Perform the conversion
tflite_model = converter.convert()


# converter=tf.lite.TFLiteConverter.from_keras_model(cnn_model)
# converter.optimizations = [tf.lite.Optimize.DEFAULT]
# tflite_model=converter.convert()

with open('guitarmidi.tflite','wb') as f:
    f.write(tflite_model)
print("TFLite model saved as guitarmidi.tflite")

Image height:  148
Before string split: (None, 148, 64), max_x=148.0


Model: "guitar_note_detector"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_spectrogram   │ (None, 148, 256,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ local_mean          │ (None, 148, 256,  │          0 │ input_spectrogra… │
│ (AveragePooling2D)  │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ local_contrast      │ (None, 148, 256,  │          0 │ input_spectrogra… │
│ (Subtract)          │ 1)                │            │ local_mean[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_to_2d       │ (None, 148, 256,  │          0 │ local_contrast[0… │
│ (Reshape)           │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_compress_conv… │ (None, 148, 64,   │        136 │ reshape_to_2d[0]… │
│ (Conv2D)            │ 8)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_compress_bn    │ (None, 148, 64,   │         32 │ freq_compress_co… │
│ (BatchNormalizatio… │ 8)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_compress_act   │ (None, 148, 64,   │          0 │ freq_compress_bn… │
│ (LeakyReLU)         │ 8)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_compress_drop  │ (None, 148, 64,   │          0 │ freq_compress_ac… │
│ (SpatialDropout2D)  │ 8)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_to_1d       │ (None, 148, 512)  │          0 │ freq_compress_dr… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ backbone_squeeze    │ (None, 148, 32)   │     16,416 │ reshape_to_1d[0]… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ backbone_squeeze_bn │ (None, 148, 32)   │        128 │ backbone_squeeze… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ backbone_squeeze_a… │ (None, 148, 32)   │          0 │ backbone_squeeze… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ backbone_conv1      │ (None, 148, 32)   │      7,200 │ backbone_squeeze… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ backbone_bn1        │ (None, 148, 32)   │        128 │ backbone_conv1[0… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ backbone_act1       │ (None, 148, 32)   │          0 │ backbone_bn1[0][… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ backbone_drop1      │ (None, 148, 32)   │          0 │ backbone_act1[0]… │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ backbone_conv2      │ (None, 148, 64)   │     14,400 │ backbone_drop1[0

 Total params: 769,364 (2.93 MB)

 Trainable params: 766,020 (2.92 MB)

 Non-trainable params: 3,344 (13.06 KB)

INFO:tensorflow:Assets written to: /tmp/tmprct9jv0g/assets


INFO:tensorflow:Assets written to: /tmp/tmprct9jv0g/assets


Saved artifact at '/tmp/tmprct9jv0g'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 148, 256, 1), dtype=tf.float32, name='input_spectrogram')
Output Type:
  TensorSpec(shape=(None, 44), dtype=tf.float32, name=None)
Captures:
  134185435738576: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134185435741840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134185435741072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134185435739920: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134185435741264: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134185435740688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134185435742032: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134185435741648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134184829370832: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134184829370640: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13418543

W0000 00:00:1774881865.873019  989328 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1774881865.873032  989328 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1774881865.873231  989328 reader.cc:83] Reading SavedModel from: /tmp/tmprct9jv0g
I0000 00:00:1774881865.875107  989328 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1774881865.875113  989328 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmprct9jv0g
I0000 00:00:1774881865.898144  989328 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1774881865.905866  989328 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1774881866.079786  989328 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmprct9jv0g
I0000 00:00:1774881866.126119  989328 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 252903 microseconds.
I0000 00:00:1774881866.180606  98932